# Cognition Engine — ARC Prize 2026 - ARC-AGI-3

Four-stage cognitive cascade: Perception → World Model → MCTS Planning → Metacognition.
No neural networks, no LLMs, pure algorithmic reasoning.

In [ ]:
# Cell 1: Install dependencies from bundled wheels (no internet needed)
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
    "--find-links", "/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
    "arc_agi", "arcengine", "pydantic", "numpy", "pillow", "python_dotenv", "requests"
])

In [ ]:
# Cell 2: Copy code to writable directory and set up paths
import os, shutil, sys

INPUT = "/kaggle/input/arc-prize-2026-arc-agi-3"
WORK  = "/kaggle/working"

# Copy agent code and cognitive module to writable location
for folder in ["ARC-AGI-3-Agents/agents", "ARC-AGI-3-Agents/cognitive"]:
    src = os.path.join(INPUT, folder)
    dst = os.path.join(WORK, os.path.basename(folder))
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

# Copy recordings dir (needed by Recorder)
os.makedirs(os.path.join(WORK, "recordings"), exist_ok=True)

# Set environment variables
os.environ["OPERATION_MODE"] = "offline"
os.environ["ENVIRONMENTS_DIR"] = os.path.join(INPUT, "environment_files")
os.environ["RECORDINGS_DIR"] = os.path.join(WORK, "recordings")
os.environ["ARC_BASE_URL"] = "https://three.arcprize.org/"
os.environ["DEBUG"] = "False"
os.environ["TESTING"] = "False"

# Add working dir to Python path so imports resolve
if WORK not in sys.path:
    sys.path.insert(0, WORK)

print("Setup complete.")

In [ ]:
# Cell 3: Run the cognitive agent on all games
import json
import logging

from agents.swarm import Swarm

# Set up logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
if not logger.handlers:
    logger.addHandler(handler)

# Discover games from local environment files
env_dir = os.environ["ENVIRONMENTS_DIR"]
games = []
for game_dir in sorted(os.listdir(env_dir)):
    game_path = os.path.join(env_dir, game_dir)
    if os.path.isdir(game_path):
        for variant in os.listdir(game_path):
            variant_path = os.path.join(game_path, variant)
            if os.path.isdir(variant_path):
                games.append(f"{game_dir}-{variant}")

print(f"Found {len(games)} games: {games}")

# Run the swarm
swarm = Swarm(
    agent="cognitiveagent",
    ROOT_URL="https://three.arcprize.org",
    games=games,
    tags=["kaggle", "cognition-engine"],
)

scorecard = swarm.main()

# Print final results
if scorecard:
    print("\n=== FINAL SCORECARD ===")
    print(json.dumps(scorecard.model_dump(), indent=2))
    print(f"\nOverall Score: {scorecard.score}")